# CRF Multi-Head — cấu hình #2 trong ablation trích xuất span (Google Colab, GPU T4)

**Bản standalone trên GitHub** của notebook 19 (dự án gốc: pipeline Shared-Boundary Multi-head cho
UIT-ViSD4SA) — cấu hình #2 trong bảng ablation 3 chiều:

| # | Cách trích xuất span | Cách gán nhãn | Notebook |
|---|---|---|---|
| 1 | CRF tuần tự (baseline) | 1 head 30-lớp `aspect#polarity` gộp | (không có trong repo này) |
| **2** | **CRF tuần tự (giữ nguyên)** | **2 head độc lập: aspect (10-lớp) + polarity (3-lớp)** | **notebook này** |
| 3 | Span enumeration + pruning | 2 head độc lập | (không có trong repo này) |

## Vì sao cần cấu hình #2
Cấu hình #2 cô lập ĐÚNG 1 biến so với 1 head 30-lớp gộp: giữ nguyên toàn bộ cơ chế CRF tuần tự
(Bước 1: embedding fusion + BiLSTM; cơ chế: Viterbi decode), chỉ tách nhãn 30-lớp gộp
(`O,B-<aspect>#<polarity>,I-<aspect>#<polarity>`) thành 2 CRF độc lập nhỏ hơn:
- **CRF aspect**: 21 tag (`O` + 10 aspect × B/I)
- **CRF polarity**: 7 tag (`O` + 3 polarity × B/I)

huấn luyện ĐỒNG THỜI qua CÙNG một BiLSTM encoder — đây chính là cơ chế "mượn kinh nghiệm" (multi-
task learning): gradient từ cả 2 loss cùng cập nhật 1 bộ trọng số encoder chung, và head polarity
học được thống kê CHUNG cho mọi aspect (vd NEUTRAL học từ CAMERA#NEUTRAL, BATTERY#NEUTRAL... khái
quát được sang DESIGN#NEUTRAL) — điều mà 1 head 30-lớp gộp không làm được (mỗi tổ hợp
`B-DESIGN#NEUTRAL` là 1 tham số riêng, không chia sẻ gì với `B-CAMERA#NEUTRAL`).

## Vì 2 CRF decode độc lập, chúng có thể lệch biên
2 CRF ở đây decode riêng biệt nên có thể cho biên span khác nhau. `merge_aspect_polarity_spans`
(`src/multihead_training.py`) xử lý việc này: dùng biên của CRF aspect làm chuẩn (aspect là "nói về
cái gì" — cấu trúc chính của bài toán), với mỗi span aspect, chọn polarity của span polarity NÀO
chồng lấn NHIỀU NHẤT (theo số ký tự); nếu không span polarity nào chồng lấn, dùng nhãn polarity đa
số của tập dữ liệu (mặc định `POSITIVE`) thay vì bỏ sót aspect span đó.

## Dữ liệu
Dùng **UIT-ViSD4SA** (Nguyen et al., PACLIC 2021, *"Span Detection for Aspect-Based Sentiment
Analysis in Vietnamese"*, https://github.com/kimkim00/UIT-ViSD4SA) — 35.396 span đã gán nhãn thủ
công trên 11.122 bình luận điện thoại tiếng Việt, theo 10 khía cạnh × 3 cực tính.

Repo này **không** commit sẵn dữ liệu đã chuyển đổi (`UIT-ViSD4SA/iob/{train,dev,test}.json`) — chỉ
có `vocab.json` (thống kê suy ra, không chứa văn bản gốc). Lý do: dữ liệu là của bên thứ 3, không có
giấy phép redistribute rõ ràng, chỉ yêu cầu trích dẫn khi dùng. Mục 2 dưới đây **tự động** tải dữ
liệu gốc + chuyển đổi (`scripts/prepare_data.py`) nếu chưa có — không cần thao tác tay.

## Hướng dẫn sử dụng
1. **Runtime → Change runtime type → GPU (T4)**.
2. Sửa `GITHUB_REPO_URL` ở Mục 2 thành URL repo GitHub thật của bạn (sau khi đã `git push`).
3. **Runtime → Run all** — Mục 2 tự `git clone` repo này + dữ liệu gốc UIT-ViSD4SA, tự chạy
   `scripts/prepare_data.py` nếu chưa có `train.json`/`dev.json`/`test.json`. Không cần upload
   file tay nào cả.
4. Cell tải PhoW2V dùng `gdown` từ mirror Drive (Mục 3).
5. Cell cuối lưu + tải kết quả xuống máy.

**Chạy cục bộ (không phải Colab)**: clone repo này + dữ liệu gốc, chạy
`python scripts/prepare_data.py --raw-dir <đường dẫn>/data --out-dir UIT-ViSD4SA/iob`, rồi mở
notebook bằng Jupyter từ thư mục `notebooks/` — Mục 2 tự dò thư mục gốc repo (không cần
`GITHUB_REPO_URL`).


## 1. Cài đặt môi trường

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
print("Đang chạy trong Google Colab:", IN_COLAB)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers", "gensim", "gdown"], check=True)

import torch
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!!! KHÔNG có GPU -- vào Runtime > Change runtime type > chọn GPU (T4) rồi chạy lại từ đầu.")


## 2. Lấy mã nguồn + dữ liệu (git clone từ GitHub, tự động)

In [ ]:
GITHUB_REPO_URL = "https://github.com/<TEN-TAI-KHOAN>/<TEN-REPO>.git"  # TODO: sửa thành URL repo thật của bạn sau khi git push

import subprocess
from pathlib import Path

if IN_COLAB:
    REPO_DIR = Path("/content/multihead-crf-uit-visd4sa")
    if not REPO_DIR.exists():
        print(f"Đang git clone {GITHUB_REPO_URL} ...")
        subprocess.run(["git", "clone", GITHUB_REPO_URL, str(REPO_DIR)], check=True)
    else:
        print("Repo đã có sẵn tại", REPO_DIR)
else:
    # Chạy cục bộ (không phải Colab): tự dò thư mục gốc repo (thư mục có "src/")
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
    print("Chạy cục bộ -- thư mục gốc repo:", REPO_DIR)

sys.path.insert(0, str(REPO_DIR))
IOB_DIR = REPO_DIR / "UIT-ViSD4SA" / "iob"
assert (IOB_DIR / "vocab.json").exists(), f"Không tìm thấy vocab.json trong {IOB_DIR} -- kiểm tra lại GITHUB_REPO_URL."
print("OK, mã nguồn tại:", REPO_DIR)

# Dữ liệu đã chuyển đổi (train.json/dev.json/test.json) KHÔNG commit sẵn (dữ liệu bên thứ ba,
# xem README.md mục "Dữ liệu") -- tự tải dữ liệu gốc UIT-ViSD4SA + chuyển đổi nếu chưa có.
if not (IOB_DIR / "train.json").exists():
    print("\nChưa có train.json/dev.json/test.json -- đang tải dữ liệu gốc UIT-ViSD4SA "
          "(Nguyen et al., PACLIC 2021) và chạy scripts/prepare_data.py...")
    RAW_DIR = Path("/content/UIT-ViSD4SA-raw") if IN_COLAB else Path("UIT-ViSD4SA-raw")
    if not RAW_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/kimkim00/UIT-ViSD4SA.git", str(RAW_DIR)], check=True)
    subprocess.run([
        sys.executable, str(REPO_DIR / "scripts" / "prepare_data.py"),
        "--raw-dir", str(RAW_DIR / "data"),
        "--out-dir", str(IOB_DIR),
    ], check=True)
    print("Đã chuyển đổi xong dữ liệu.")
else:
    print("Đã có sẵn dữ liệu đã chuyển đổi tại", IOB_DIR)


## 3. Tải PhoW2V syllable embedding (458MB, mirror Google Drive)

Y hệt notebook 16/18 -- cùng nguồn PhoW2V, cùng vocab.

In [ ]:
PHOW2V_DIR = Path("/content/phow2v") if IN_COLAB else Path("phow2v")
PHOW2V_DIR.mkdir(exist_ok=True)
PHOW2V_FOLDER_URL = "https://drive.google.com/drive/folders/1NZhZFYbcwKzLpvvGdJUdPbwEVdVW4E3j"

zip_candidates = list(PHOW2V_DIR.rglob("*syllables_100dims*.zip")) or list(PHOW2V_DIR.rglob("*.zip"))
if IN_COLAB and not zip_candidates:
    import gdown
    try:
        gdown.download_folder(url=PHOW2V_FOLDER_URL, output=str(PHOW2V_DIR), quiet=False, use_cookies=False)
    except Exception as e:
        print(f"gdown tải tự động thất bại ({e}).")
        print(f"  1. Mở {PHOW2V_FOLDER_URL} trên trình duyệt, tải 'word2vec_vi_syllables_100dims.zip' về máy.")
        print("  2. Chạy lại cell này -- nó sẽ hỏi upload file vừa tải.")
        from google.colab import files
        uploaded = files.upload()
        for name in uploaded:
            Path(name).rename(PHOW2V_DIR / name)

zip_candidates = list(PHOW2V_DIR.rglob("*syllables_100dims*.zip")) or list(PHOW2V_DIR.rglob("*.zip"))
assert zip_candidates, f"Không tìm thấy file .zip nào trong {PHOW2V_DIR}"
PHOW2V_ZIP = zip_candidates[0]
print("Đã có:", PHOW2V_ZIP, f"({PHOW2V_ZIP.stat().st_size / 1e6:.1f} MB)")

PHOW2V_EXTRACT_DIR = PHOW2V_DIR / "extracted"
with zipfile.ZipFile(PHOW2V_ZIP) as zf:
    zf.extractall(PHOW2V_EXTRACT_DIR)
print("Đã giải nén vào:", PHOW2V_EXTRACT_DIR)


## 4. Nạp vocab + xây ma trận syllable embedding từ PhoW2V

In [ ]:
from src.span_dataset import load_vocab
from src.pretrained_syllable_embedding import load_word2vec_vectors, build_syllable_embedding_matrix

vocab = load_vocab(IOB_DIR / "vocab.json")
print(f"Syllable vocab: {len(vocab['syllable']):,} | Char vocab: {len(vocab['char']):,}")
print(f"Tag vocab -- aspect: {len(vocab['tag']['aspect'])} | polarity: {len(vocab['tag']['polarity'])} "
      f"| aspect_polarity (chỉ dùng để decode gold): {len(vocab['tag']['aspect_polarity'])}")

kv = load_word2vec_vectors(PHOW2V_EXTRACT_DIR)
print(f"PhoW2V: {len(kv.index_to_key):,} từ, dim={kv.vector_size}")

syllable_matrix, coverage_stats = build_syllable_embedding_matrix(vocab["syllable"], kv)
print(f"Coverage: {coverage_stats['coverage']:.1%} ({coverage_stats['n_found']:,}/{coverage_stats['n_real_tokens']:,})")


## 5. Dataset multi-head + dynamic token-budget batching

Dùng `UIT-ViSD4SA/iob/{train,dev,test}.json` -- CÙNG nguồn dữ liệu với baseline (notebook 16), chỉ khác `MultiHeadCollator` đọc cả 3 trường tag (`tags_aspect`/`tags_polarity`/`tags_aspect_polarity`) thay vì 1 scheme.

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.multihead_dataset import MultiHeadCollator, MultiHeadSpanDataset
from src.token_batch_sampler import TokenBudgetBatchSampler

TOKEN_BUDGET = 5000  # đồng bộ với baseline (NB16, Cập nhật v7/v9) -- src/multihead_training.py
                      # có CÙNG cơ chế an toàn (fp16 + gradient checkpointing + tự bỏ qua batch
                      # OOM), nên nâng lên 5000 y hệt baseline để 2 cấu hình chỉ khác đúng 1 biến
                      # (kiến trúc CRF), không lệch thêm token_budget khi so sánh
CONTEXTUAL_MODEL_NAME = "xlm-roberta-large"  # cấu hình tốt nhất bài báo gốc, y hệt baseline
tokenizer = AutoTokenizer.from_pretrained(CONTEXTUAL_MODEL_NAME)

train_ds = MultiHeadSpanDataset(IOB_DIR / "train.json")
dev_ds = MultiHeadSpanDataset(IOB_DIR / "dev.json")
test_ds = MultiHeadSpanDataset(IOB_DIR / "test.json")
print(f"train={len(train_ds):,}  dev={len(dev_ds):,}  test={len(test_ds):,}")

collate = MultiHeadCollator(vocab, tokenizer=tokenizer, use_contextual=True)

train_lengths = [len(d["tokens"]) for d in train_ds.docs]
dev_lengths = [len(d["tokens"]) for d in dev_ds.docs]
test_lengths = [len(d["tokens"]) for d in test_ds.docs]

train_loader = DataLoader(train_ds, collate_fn=collate,
                          batch_sampler=TokenBudgetBatchSampler(train_lengths, token_budget=TOKEN_BUDGET, shuffle=True, seed=42))
dev_loader = DataLoader(dev_ds, collate_fn=collate,
                         batch_sampler=TokenBudgetBatchSampler(dev_lengths, token_budget=TOKEN_BUDGET, shuffle=False))
test_loader = DataLoader(test_ds, collate_fn=collate,
                          batch_sampler=TokenBudgetBatchSampler(test_lengths, token_budget=TOKEN_BUDGET, shuffle=False))
print(f"Token budget/batch = {TOKEN_BUDGET:,}")
print("Nếu log cho thấy nhiều batch bị bỏ qua vì OOM, giảm TOKEN_BUDGET rồi chạy lại từ cell này.")


## 6-7. Dựng model + huấn luyện (đa-seed)

Step 1 (embedding fusion + BiLSTM) và siêu tham số y hệt baseline (`16_final_pipeline_colab.ipynb`)
-- chỉ khác `num_tags_aspect`/`num_tags_polarity` thay cho `num_tags` gộp.

**Đa-seed (mean±std)**: `SEEDS` mặc định chỉ có `42` (giữ tốc độ "Run all" như trước). Bỏ comment
dòng `SEEDS = [42, 123, 2024, 777]` để chạy đủ 4 lần độc lập -- mỗi lần dựng model MỚI (seed khác
nhau ảnh hưởng cả init trọng số lẫn thứ tự shuffle batch), lưu `results_multihead_crf_seed<seed>.json`
RIÊNG, rồi tổng hợp mean/std ở Mục 9. Model của mỗi seed bị giải phóng khỏi VRAM ngay sau khi huấn
luyện xong seed đó (không giữ nhiều bản 564M-tham số cùng lúc).

Mỗi lần chạy còn lưu `test_predictions_multihead_seed<seed>.jsonl` -- span dự đoán thô từng câu của
CẢ 2 nhánh (aspect, polarity) lẫn sau khi hợp nhất, dùng cho `scripts/bootstrap_compare_macro_f1.py`.

In [ ]:
import json
import time

from src.multihead_model import BiLSTMMultiHeadCRFTagger
from src.multihead_training import predict_dataset, train_model, aggregate_merge_stats
from src.evaluation import evaluate, save_raw_predictions
from src.training import set_seed

EPOCHS = 30
PATIENCE = 5
LR = 1e-3
XLMR_LR = 2e-5  # y hệt baseline (cùng cơ chế CRF, giữ cùng optimizer setup để so sánh công bằng)
WEIGHT_DECAY = 1e-2  # áp cho 2 CRF (Eq. 11 bài báo), phần còn lại weight_decay=0 -- y hệt baseline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Mặc định chỉ chạy seed=42 (giữ tốc độ "Run all" như cũ). Bỏ comment dòng dưới để chạy đủ 4 seed.
SEEDS = [42]
# SEEDS = [42, 123, 2024, 777]

all_results = {}          # all_results[seed] = results_payload dict
all_checkpoints = {}       # all_checkpoints[seed] = checkpoint_path
all_merge_stats = {}       # all_merge_stats[seed] = aggregate_merge_stats(...) cho seed đó
all_results_paths = {}     # all_results_paths[seed] = đường dẫn file JSON đã lưu

for seed in SEEDS:
    print(f"\n{'#'*70}\nSEED = {seed}\n{'#'*70}")
    set_seed(seed)  # seed CẢ random/numpy/torch-CPU/torch-CUDA, trước khi dựng model + DataLoader

    model = BiLSTMMultiHeadCRFTagger(
        syllable_vocab_size=len(vocab["syllable"]),
        char_vocab_size=len(vocab["char"]),
        num_tags_aspect=len(vocab["tag"]["aspect"]),
        num_tags_polarity=len(vocab["tag"]["polarity"]),
        use_char=True, use_contextual=True,
        contextual_model_name=CONTEXTUAL_MODEL_NAME,
        contextual_projected_dim=100,
        lstm_hidden=400, dropout=0.33,
        pretrained_syllable_matrix=syllable_matrix,
        freeze_syllable=False,    # PhoW2V FINE-TUNE, y hệt baseline (NB16 Cập nhật v11)
        freeze_contextual=False,  # XLM-R FINE-TUNE, y hệt baseline (NB16 Cập nhật v11)
    ).to(device)

    n_total = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Tổng tham số: {n_total:,} | Trainable: {n_trainable:,} ({n_trainable/n_total:.1%})")

    checkpoint_path = (Path("/content") if IN_COLAB else Path(".")) / f"best_model_multihead_seed{seed}.pt"
    t0 = time.time()
    result = train_model(
        model, train_loader, dev_loader, vocab, device,
        epochs=EPOCHS, lr=LR, xlmr_lr=XLMR_LR, weight_decay=WEIGHT_DECAY,
        patience=PATIENCE, checkpoint_path=checkpoint_path,
    )
    print(f"Thời gian train (seed={seed}): {(time.time() - t0) / 60:.1f} phút")
    print(f"Dev macro-F1 tốt nhất (seed={seed}): {result['best_dev_macro_f1']:.4f}")

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    # collect_details=True -- lấy thêm span thô từng câu (aspect/polarity/merged/gold) + merge_stats
    # từng câu, không chỉ tp/fp/fn tổng hợp.
    gold_test, pred_test, _, test_details = predict_dataset(model, test_loader, vocab, device, collect_details=True)
    test_metrics = evaluate(gold_test, pred_test)
    print(f"TEST (seed={seed}) -> micro F1={test_metrics['micro']['f1']:.4f}  macro F1={test_metrics['macro']['f1']:.4f}")
    print(f"Bài báo gốc (aspect_polarity, syllable+char+XLM-R-Large): F1-macro = 45.70%")

    predictions_path = (Path("/content") if IN_COLAB else Path(".")) / f"test_predictions_multihead_seed{seed}.jsonl"
    save_raw_predictions(test_details, predictions_path)
    print(f"Đã lưu span thô từng câu: {predictions_path} ({len(test_details):,} câu)")

    merge_stats_seed = aggregate_merge_stats([d["merge_stats"] for d in test_details])
    all_merge_stats[seed] = merge_stats_seed

    results_payload = {
        "config": {
            "model": "BiLSTM + 2-CRF-head (aspect + polarity)", "scheme": "aspect_polarity", "seed": seed,
            "epochs_ran": len(result["history"]), "epochs_max": EPOCHS,
            "token_budget": TOKEN_BUDGET, "lr": LR, "xlmr_lr": XLMR_LR, "weight_decay": WEIGHT_DECAY,
            "patience": PATIENCE, "lstm_hidden": 400, "dropout": 0.33,
            "contextual_model": CONTEXTUAL_MODEL_NAME, "contextual_projected_dim": 100,
            "syllable_source": "PhoW2V_syllables_100dims",
            "num_tags_aspect": len(vocab["tag"]["aspect"]), "num_tags_polarity": len(vocab["tag"]["polarity"]),
            "n_total_params": n_total, "n_trainable_params": n_trainable,
        },
        "history": result["history"],
        "best_dev_macro_f1": result["best_dev_macro_f1"],
        "test_micro": test_metrics["micro"],
        "test_macro": test_metrics["macro"],
        "test_per_class": test_metrics["per_class"],
        "merge_stats": merge_stats_seed,
    }

    results_path = (Path("/content") if IN_COLAB else Path(".")) / f"results_multihead_crf_seed{seed}.json"
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(results_payload, f, ensure_ascii=False, indent=2)
    print("Đã lưu:", results_path)
    if IN_COLAB:
        files.download(str(results_path))
        files.download(str(checkpoint_path))
        files.download(str(predictions_path))

    all_results[seed] = results_payload
    all_checkpoints[seed] = checkpoint_path
    all_results_paths[seed] = results_path

    del model  # giải phóng VRAM trước khi sang seed tiếp theo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# "results_path" trỏ tới seed ĐẦU TIÊN (mặc định 42) -- các cell so sánh bên dưới (Mục 11) dùng
# đúng 1 file làm đại diện, giống hành vi cũ trước khi có đa-seed.
results_path = all_results_paths[SEEDS[0]]
result = all_results[SEEDS[0]]


## 8. Thống kê bước hợp nhất `merge_aspect_polarity_spans` (Thành phần bắt buộc)

2 tỉ lệ chẩn đoán trực tiếp cho cơ chế hợp nhất 2 CRF (Mục III.C bài): tỉ lệ span aspect phải nhận
cực tính MẶC ĐỊNH (không polarity span nào chồng lấn -- dấu hiệu 2 head bất đồng ranh giới), và tỉ
lệ polarity span "đơn độc" bị loại (chưa từng là lựa chọn tốt nhất cho aspect span nào). Tính từ
`merge_stats` đã lưu trong mỗi `results_multihead_crf_seed<seed>.json` ở Mục 6-7 (không cần chạy
lại mạng nơ-ron, chỉ hậu xử lý trên span đã dự đoán sẵn).

In [ ]:
for seed in SEEDS:
    s = all_merge_stats[seed]
    print(f"--- seed={seed} ---")
    print(f"Tỉ lệ span nhận cực tính mặc định: {s['default_assigned_rate_pct']:.2f}% "
          f"({s['default_assigned']:,}/{s['total_aspect_spans']:,} span aspect)")
    print(f"Tỉ lệ polarity span đơn độc bị loại: {s['orphan_polarity_discarded_rate_pct']:.2f}% "
          f"({s['orphan_polarity_discarded']:,}/{s['total_polarity_spans']:,} span polarity)")


## 9. (Sau khi có đủ nhiều seed) Tổng hợp mean ± std qua nhiều lần chạy

Đọc lại các file `results_multihead_crf_seed<seed>.json` đã lưu ở Mục 6-7 (từ phiên này hoặc phiên
Colab khác đã chạy trước) -- không phụ thuộc biến trong bộ nhớ. Cần ít nhất 2 seed để std có ý
nghĩa; lý tưởng đủ 4 seed (`42, 123, 2024, 777`) như Mục 6-7 đã set up.

In [ ]:
import json as _json2

import numpy as np
import pandas as pd

ALL_SEEDS_TO_AGGREGATE = [42, 123, 2024, 777]  # sửa lại đúng danh sách seed đã thực sự chạy

seed_results_agg = {}
for seed in ALL_SEEDS_TO_AGGREGATE:
    p = (Path("/content") if IN_COLAB else Path(".")) / f"results_multihead_crf_seed{seed}.json"
    if p.exists():
        with open(p, encoding="utf-8") as f:
            seed_results_agg[seed] = _json2.load(f)
    else:
        print(f"Chưa có {p} -- bỏ qua seed {seed} (chạy Mục 6-7 với seed này trước, hoặc upload file nếu chạy ở phiên khác).")

if len(seed_results_agg) < 2:
    print(f"\nMới có {len(seed_results_agg)}/{len(ALL_SEEDS_TO_AGGREGATE)} seed -- cần ít nhất 2 để tính std có ý nghĩa.")
else:
    macro_f1_runs = np.array([r["test_macro"]["f1"] for r in seed_results_agg.values()]) * 100
    print(f"Số seed đã có: {len(seed_results_agg)}/{len(ALL_SEEDS_TO_AGGREGATE)} ({list(seed_results_agg.keys())})")
    print(f"Mean macro-F1: {macro_f1_runs.mean():.2f}%")
    print(f"Std macro-F1:  {macro_f1_runs.std(ddof=1):.2f}pp")

    per_class_runs = {}
    for r in seed_results_agg.values():
        for label, m in r["test_per_class"].items():
            per_class_runs.setdefault(label, []).append(m["f1"])
    per_class_summary = {
        label: {
            "mean_f1_pct": float(np.mean(f1s)) * 100,
            "std_f1_pct": float(np.std(f1s, ddof=1)) * 100 if len(f1s) > 1 else None,
            "n_seeds": len(f1s),
        }
        for label, f1s in per_class_runs.items()
    }
    print("\n=== Mean/std F1 theo từng lớp aspect#polarity, qua các seed đã có ===")
    display(pd.DataFrame(per_class_summary).T.round(2).sort_values("mean_f1_pct"))


## 11. So sánh cả 3 cấu hình (baseline / CRF multi-head / Span-ViSD)

Upload `results_aspect_polarity_final_pipeline_seed42.json` (NB16) và/hoặc `results_span_visd.json`
(NB18) để so trực tiếp; bỏ qua file nào chưa có. So sánh dùng kết quả seed đầu tiên (`SEEDS[0]`,
mặc định 42) làm đại diện -- xem Mục 9 để so mean±std qua nhiều seed thay vì 1 lần chạy đơn lẻ.

In [ ]:
from src.experiment_tracking import rare_class_report, summarize_experiments

BASELINE_RESULTS_NAME = "results_aspect_polarity_final_pipeline_seed42.json"
SPAN_VISD_RESULTS_NAME = "results_span_visd.json"

experiments = {"CRF multi-head (NB19, cấu hình #2)": str(results_path)}

if IN_COLAB:
    print(f"Upload file kết quả BASELINE ({BASELINE_RESULTS_NAME}, từ notebook 16), hoặc bỏ qua (Cancel):")
    uploaded_baseline = files.upload()
    if uploaded_baseline:
        experiments["baseline (NB16, cấu hình #1, 1 head CRF)"] = next(iter(uploaded_baseline))

    print(f"Upload file kết quả Span-ViSD ({SPAN_VISD_RESULTS_NAME}, từ notebook 18), hoặc bỏ qua (Cancel):")
    uploaded_span_visd = files.upload()
    if uploaded_span_visd:
        experiments["Span-ViSD (NB18, cấu hình #3, span-enumeration + 2 head)"] = next(iter(uploaded_span_visd))
else:
    if Path(BASELINE_RESULTS_NAME).exists():
        experiments["baseline (NB16, cấu hình #1, 1 head CRF)"] = BASELINE_RESULTS_NAME
    if Path(SPAN_VISD_RESULTS_NAME).exists():
        experiments["Span-ViSD (NB18, cấu hình #3, span-enumeration + 2 head)"] = SPAN_VISD_RESULTS_NAME

if len(experiments) > 1:
    comparison = summarize_experiments(experiments)
    print("=== So sánh tổng quan (3 cấu hình, nếu có đủ file) ===")
    display(comparison[["label", "test_micro_f1", "test_macro_f1", "gap_vs_paper",
                         "rare_class_macro_f1", "n_rare_classes_f1_zero", "train_time_min"]].round(4))

    print("\n=== F1 các lớp cực hiếm -- CRF multi-head (NB19) ===")
    display(rare_class_report(str(results_path))[["label", "precision", "recall", "f1", "tp", "fp", "fn"]])
else:
    print("Chưa có file nào khác để so sánh ngay -- so các file JSON kết quả sau bằng src/experiment_tracking.py.")


## Kết luận

- Đã huấn luyện cấu hình #2 (CRF multi-head: encoder + cơ chế trích xuất y hệt baseline, chỉ tách
  nhãn thành 2 CRF độc lập aspect+polarity), đánh giá bằng đúng 1 hàm `evaluate()` với baseline và
  Span-ViSD để so sánh công bằng cả 3 cấu hình.
- So `#2` với `#1` (baseline) cô lập ĐÚNG hiệu ứng của ý tưởng multi-head (không đổi cách trích
  xuất span) -- nếu `#2` cải thiện `rare_class_macro_f1`/`n_rare_classes_f1_zero` so với `#1`, đó
  là bằng chứng trực tiếp cho luận điểm "tách 2 head giúp polarity mượn kinh nghiệm giữa các
  aspect", không bị nhiễu bởi việc đổi kiến trúc trích xuất như khi so `#3` với `#1`.
- **Giới hạn đã biết**: 2 CRF decode độc lập có thể lệch biên; `merge_aspect_polarity_spans` dùng
  biên của CRF aspect làm chuẩn và chọn polarity theo overlap lớn nhất -- đây là 1 lựa chọn thiết
  kế cụ thể (không phải cách duy nhất có thể), nên khi phân tích lỗi nên kiểm tra riêng xem lỗi đến
  từ CRF aspect, CRF polarity, hay bước hợp nhất. **Mục 8** giờ đo trực tiếp 2 tỉ lệ này (span nhận
  cực tính mặc định, polarity span đơn độc bị loại) thay vì chỉ suy đoán định tính.
- **Đa-seed + span thô**: mỗi lần chạy Mục 6-7 giờ lưu thêm `test_predictions_multihead_seed
  <seed>.jsonl` (span thô từng câu của cả 2 nhánh lẫn sau hợp nhất) và hỗ trợ chạy nhiều seed độc
  lập (`SEEDS`) để báo cáo macro-F1 dạng mean±std (Mục 9) thay vì 1 số duy nhất -- xem thảo luận
  trước đó trong dự án về phương sai run-to-run (~1 điểm F1 chỉ do GPU non-determinism/OOM-skip).